# Solutions - Manipulation de données avec Pandas

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Pour afficher plus de colonnes dans les DataFrames
pd.set_option('display.max_columns', 30)

In [ ]:
# Chargement du jeu de données
df = pd.read_csv('../../data/passenger_satisfaction/train_50.csv')

# Affichage des premières lignes
df.head()

## Exercice 1
Calculez l'âge moyen des passagers par classe et par genre. Affichez les résultats sous forme de tableau.

In [ ]:
# Calcul de l'âge moyen par classe et par genre
age_by_class_gender = df.groupby(['Class', 'Gender'])['Age'].mean().unstack()

# Affichage du tableau
print("Âge moyen des passagers par classe et par genre:")
print(age_by_class_gender)

# Visualisation avec Plotly
fig = px.bar(
    age_by_class_gender.reset_index().melt(id_vars='Class', var_name='Gender', value_name='Average Age'),
    x='Class',
    y='Average Age',
    color='Gender',
    barmode='group',
    title='Âge moyen par classe et par genre',
    labels={'Average Age': 'Âge moyen', 'Class': 'Classe'}
)
fig.show()

## Exercice 2
Créez une nouvelle colonne 'Long_Flight' qui vaut True si la distance de vol est supérieure à 1000 km, et False sinon. Puis calculez le taux de satisfaction pour les vols courts et les vols longs.

In [ ]:
# Création de la colonne Long_Flight
df['Long_Flight'] = df['Flight Distance'] > 1000

# Affichage des premiers résultats pour vérification
df[['Flight Distance', 'Long_Flight']].head(10)

In [ ]:
# Calcul du taux de satisfaction pour les vols courts et longs
satisfaction_by_flight_length = pd.crosstab(df['Long_Flight'], df['Satisfaction'], normalize='index') * 100
print("Taux de satisfaction selon la longueur du vol (%) :")
print(satisfaction_by_flight_length)

# Préparation des données pour Plotly
flight_length_df = satisfaction_by_flight_length.reset_index()
flight_length_df['Long_Flight'] = flight_length_df['Long_Flight'].map({False: 'Vol court (<1000 km)', True: 'Vol long (>1000 km)'})

# Visualisation avec Plotly
fig = px.bar(
    flight_length_df, 
    x='Long_Flight', 
    y='satisfied',
    title='Taux de satisfaction selon la longueur du vol',
    labels={'satisfied': 'Pourcentage de passagers satisfaits (%)', 'Long_Flight': 'Longueur du vol'},
    color='Long_Flight',
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.update_layout(height=500, width=700)
fig.show()

## Exercice 3
Identifiez les 3 services qui ont le plus grand écart de score entre les passagers satisfaits et insatisfaits. Ces services pourraient être les plus importants pour améliorer la satisfaction globale.

In [ ]:
# Identification des colonnes de services
rating_columns = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Calcul des scores moyens par service selon la satisfaction
ratings_by_satisfaction = df.groupby('Satisfaction')[rating_columns].mean()

# Calcul de l'écart de score pour chaque service
score_diff = ratings_by_satisfaction.loc['satisfied'] - ratings_by_satisfaction.loc['neutral or dissatisfied']
score_diff = score_diff.sort_values(ascending=False)

# Affichage des 3 services avec le plus grand écart
print("Les 3 services avec le plus grand écart de score entre passagers satisfaits et insatisfaits:")
top_3_services = score_diff.head(3)
print(top_3_services)

# Visualisation des écarts pour tous les services
score_diff_df = score_diff.reset_index()
score_diff_df.columns = ['Service', 'Score Difference']

fig = px.bar(
    score_diff_df,
    x='Service',
    y='Score Difference',
    title='Écart de score entre passagers satisfaits et insatisfaits par service',
    labels={'Score Difference': 'Écart de score', 'Service': 'Service'},
    color='Score Difference',
    color_continuous_scale=px.colors.sequential.Viridis
)
fig.update_layout(
    xaxis={'categoryorder':'total descending'},
    xaxis_tickangle=-45,
    height=600,
    width=900
)
fig.show()

## Analyse supplémentaire : Corrélation entre les variables et la satisfaction

In [ ]:
# Création d'une variable binaire pour la satisfaction
df['Satisfied_Binary'] = df['Satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Sélection des colonnes numériques pour la corrélation
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'id']

# Calcul des corrélations avec la satisfaction
correlations = df[numeric_cols].corr()['Satisfied_Binary'].sort_values(ascending=False)

# Affichage des corrélations les plus fortes (positives et négatives)
print("Variables les plus corrélées positivement avec la satisfaction:")
print(correlations.head(5))
print("\nVariables les plus corrélées négativement avec la satisfaction:")
print(correlations.tail(5))

# Visualisation des corrélations
corr_df = pd.DataFrame({'Variable': correlations.index, 'Correlation': correlations.values})
corr_df = corr_df[corr_df['Variable'] != 'Satisfied_Binary']
corr_df = corr_df.sort_values('Correlation')

fig = px.bar(
    corr_df,
    x='Correlation',
    y='Variable',
    orientation='h',
    title='Corrélation des variables avec la satisfaction',
    color='Correlation',
    color_continuous_scale=px.colors.diverging.RdBu,
    range_color=[-1, 1]
)
fig.update_layout(height=800, width=800)
fig.show()